### PySpark Otomoto Demo 

Źródło danych: https://www.kaggle.com/datasets/szymoncyperski/car-sales-offers-from-otomotopl-2023 


In [2]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'pyspark'

**Teoria:** Powyżej importujemy niezbędne biblioteki. `SparkSession` to główny punkt wejścia do funkcjonalności DataFrame i SQL w Sparku (od wersji 2.0). Moduł `functions` dostarcza wbudowane funkcje operujące na kolumnach, a `matplotlib.pyplot` posłuży nam do późniejszej wizualizacji danych.


In [2]:
spark = SparkSession.builder \
    .appName("Otomoto Demo") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/12 18:28:09 WARN Utils: Your hostname, MacBook-Air-Remigiusz.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.2 instead (on interface en0)
26/06/12 18:28:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 18:28:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


**Teoria:** Tworzymy sesję Sparka. `builder` używa wzorca projektowego Builder do skonfigurowania sesji. `getOrCreate()` tworzy nową sesję lub pobiera istniejącą, co jest bezpieczne przy wielokrotnym uruchamianiu notatnika.


In [7]:
df = spark.read.option("header", True) \
    .option("delimiter", ";") \
    .option("inferSchema", False) \
    .csv("otomoto_offers_eng_23-04-2023.csv")


26/06/12 18:28:43 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: otomoto_offers_eng_23-04-2023.csv.
java.io.FileNotFoundException: File otomoto_offers_eng_23-04-2023.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.s

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/remigiuszkrupa/Desktop/LabPython/LAB-ARR/Lab2/AAP_LAB_TASKS/materialy/lab_05/otomoto_offers_eng_23-04-2023.csv. SQLSTATE: 42K03

**Teoria:** Wczytywanie danych. Spark używa leniwego ewaluowania (lazy evaluation) - dane nie są fizycznie wczytywane w tym momencie, tworzony jest tylko plan wykonania (DAG). Ustawiamy `header=True` ponieważ nasz plik CSV ma nagłówki, oraz określamy separator jako średnik `;`.


In [6]:
df.show()

NameError: name 'df' is not defined

**Teoria:** `show()` to akcja (action), która uruchamia wykonanie obliczeń w Sparku. Dopiero teraz plik jest odczytywany, a wynik prezentowany na ekranie.


In [5]:
df.filter(F.col("vehicle_brand") == "Volvo").show()

NameError: name 'df' is not defined

In [4]:
df = df.withColumn("price_num",
                   F.regexp_replace(F.col("price"), r"[^\d]", "").cast("double"))

df = df.withColumn("mileage_km",
                   F.regexp_replace(F.col("mileage"), r"[^\d]", "").cast("integer"))

df = df.withColumn("production_year_int",
                   F.regexp_replace(F.col("production_year"), r"[^\d]", "").cast("integer"))

df = df.withColumn("engine_cc",
                   F.regexp_replace(F.col("engine_displacement"), r"[^\d]", "").cast("integer"))

df = df.withColumn("power_hp",
                   F.regexp_replace(F.col("power"), r"[^\d]", "").cast("integer"))

df = df.withColumn("fuel_clean",
                   F.lower(F.trim(F.col("fuel_type"))))

NameError: name 'df' is not defined

In [ ]:
df.select("vehicle_brand", "vehicle_model", "price_num", "mileage_km",
          "production_year_int", "engine_cc", "power_hp", "fuel_clean") \
  .show(10, truncate=False)

+-------------+-------------+---------+----------+-------------------+---------+--------+--------------+
|vehicle_brand|vehicle_model|price_num|mileage_km|production_year_int|engine_cc|power_hp|fuel_clean    |
+-------------+-------------+---------+----------+-------------------+---------+--------+--------------+
|Volvo        |V70          |23200.0  |304000    |2010               |15603    |109     |diesel        |
|Honda        |Accord       |16800.0  |236000    |2005               |19983    |155     |gasoline      |
|Mercedes-Benz|Klasa X      |249900.0 |73000     |2019               |29873    |258     |diesel        |
|Toyota       |Avensis      |16499.0  |220000    |2005               |17943    |129     |gasoline      |
|Ford         |C-MAX        |29900.0  |179058    |2012               |19973    |140     |diesel        |
|Peugeot      |208          |80900.0  |1         |2023               |11993    |75      |gasoline      |
|Kia          |Sportage     |74770.0  |100420    |2018 

**Teoria:** `select()` to transformacja, która działa jak w SQL - pozwala wybrać podzbiór kolumn. Zmniejsza to ilość przetwarzanych danych w dalszych krokach.


In [ ]:
avg_brand = df.groupBy("vehicle_brand") \
              .agg(F.round(F.avg("price_num"), 2).alias("avg_price")) \
              .orderBy(F.col("avg_price").desc())

print("Średnia cena per marka")
avg_brand.show(20, truncate=False)

Średnia cena per marka


+-------------+----------+
|vehicle_brand|avg_price |
+-------------+----------+
|Lamborghini  |1281601.59|
|Ferrari      |1015648.41|
|McLaren      |866462.12 |
|Rolls-Royce  |762649.96 |
|Bentley      |657051.57 |
|Maybach      |575529.83 |
|Aston Martin |530891.29 |
|KTM          |389000.0  |
|Porsche      |363902.5  |
|Alpine       |352555.5  |
|RAM          |348905.66 |
|Geely        |346903.0  |
|BMW-ALPINA   |333006.94 |
|LEVC         |305776.0  |
|Skywell      |274743.33 |
|Maxus        |255052.59 |
|Tesla        |232544.92 |
|Caterham     |227550.0  |
|Land Rover   |200495.64 |
|Maserati     |197591.6  |
+-------------+----------+
only showing top 20 rows


In [ ]:
fuel_count = df.groupBy("fuel_clean").count()
print("Liczba ogłoszeń wg rodzaju paliwa")
fuel_count.show()

Liczba ogłoszeń wg rodzaju paliwa
+--------------+------+
|    fuel_clean| count|
+--------------+------+
|      gasoline|101165|
|        diesel| 88624|
|gasoline + lpg|  7419|
|        hybrid|  7527|
|gasoline + cng|    96|
|      electric|  3373|
|      hydrogen|     1|
+--------------+------+



In [ ]:
df.createOrReplaceTempView("cars")

In [ ]:
# SQL: zależność mocy i pojemności od ceny
spark.sql("""
    SELECT vehicle_brand,
           ROUND(AVG(power_hp), 1) AS avg_power,
           ROUND(AVG(engine_cc), 1) AS avg_cc,
           ROUND(AVG(price_num), 1) AS avg_price
    FROM cars
    GROUP BY vehicle_brand
    ORDER BY avg_power DESC
""").show()

+-------------+---------+-------+---------+
|vehicle_brand|avg_power| avg_cc|avg_price|
+-------------+---------+-------+---------+
|      McLaren|    652.5|38811.5| 866462.1|
|  Lamborghini|    637.8|51767.4|1281601.6|
|      Ferrari|    632.9|46168.3|1015648.4|
|      Bentley|    550.8|54644.1| 657051.6|
|      Maybach|    541.0|53499.7| 575529.8|
| Aston Martin|    528.1|48457.7| 530891.3|
|   BMW-ALPINA|    516.0|40259.4| 333006.9|
|        Tesla|    486.4| 6023.0| 232544.9|
|  Rolls-Royce|    479.7|65997.4| 762650.0|
|          RAM|    413.2|55898.3| 348905.7|
|     Polestar|    408.0|   NULL| 146750.0|
|     Maserati|    397.7|31911.3| 197591.6|
|      Porsche|    376.8|33072.8| 363902.5|
|          KTM|    362.0|19843.0| 389000.0|
|        Dodge|    341.5|46547.9| 142202.8|
|      Genesis|    336.3|31915.5| 108725.0|
|     Cadillac|    309.1|47372.8| 140824.6|
|          GMC|    290.3|50369.8| 135253.3|
|     Plymouth|    287.5|50298.0| 125230.0|
|       Hummer|    278.3|49413.6

In [ ]:
df.groupBy("production_year_int") \
  .count() \
  .orderBy(F.col("production_year_int").desc()) \
  .show()

+-------------------+-----+
|production_year_int|count|
+-------------------+-----+
|               2023|12987|
|               2022|13680|
|               2021| 7729|
|               2020| 7787|
|               2019|13978|
|               2018|15096|
|               2017|13952|
|               2016|11755|
|               2015|10640|
|               2014| 9988|
|               2013| 9341|
|               2012| 9664|
|               2011|10108|
|               2010| 9115|
|               2009| 9040|
|               2008| 8846|
|               2007| 7816|
|               2006| 6320|
|               2005| 5121|
|               2004| 3647|
+-------------------+-----+
only showing top 20 rows


In [ ]:
# Średnia cena i przebieg per marka i model
df.groupBy("vehicle_brand", "vehicle_model") \
  .agg(
      F.round(F.avg("price_num"), 2).alias("avg_price"),
      F.round(F.avg("mileage_km"), 2).alias("avg_mileage")
  ) \
  .orderBy(F.col("avg_price").desc()) \
  .show(20, truncate=False)

+-------------+-------------+----------+-----------+
|vehicle_brand|vehicle_model|avg_price |avg_mileage|
+-------------+-------------+----------+-----------+
|Ferrari      |812 GTS      |2750000.0 |2000.0     |
|Mercedes-Benz|SLR          |2340000.0 |2380.0     |
|Ferrari      |SF90 Stradale|2293626.0 |1855.75    |
|Rolls-Royce  |Dawn         |1999999.0 |4300.0     |
|Lamborghini  |Aventador    |1822661.67|14467.5    |
|Ferrari      |812 Superfast|1670200.0 |16414.0    |
|Lamborghini  |Murcielago   |1550000.0 |46950.0    |
|McLaren      |Artura       |1499000.0 |2100.0     |
|McLaren      |675Lt        |1290000.0 |18500.0    |
|McLaren      |600lt-coupe  |1289000.0 |8500.0     |
|Lamborghini  |Huracan      |1258737.0 |25671.46   |
|Rolls-Royce  |Phantom      |1225000.0 |22000.0    |
|Rolls-Royce  |Ghost        |1206316.67|39451.67   |
|Bentley      |Bentayga     |1200222.11|34022.72   |
|Lamborghini  |Diablo       |1199900.0 |32164.0    |
|Rolls-Royce  |Wraith       |1183000.0 |54333.

In [ ]:
# zależność ceny od przebiegu 
price_mileage = df.select("price_num", "mileage_km") \
                  .where((F.col("price_num").isNotNull()) & (F.col("mileage_km").isNotNull()))

In [3]:
pdf_scatter = price_mileage.sample(fraction=0.1, seed=42).toPandas()

plt.figure(figsize=(8,5))
plt.scatter(pdf_scatter["mileage_km"], pdf_scatter["price_num"], s=6)
plt.title("Cena vs Przebieg")
plt.xlabel("Przebieg [km]")
plt.ylabel("Cena")
plt.tight_layout()
plt.savefig("scatter_price_mileage.png")

print("Wizualizacja scatter zapisana jako scatter_price_mileage.png")

NameError: name 'price_mileage' is not defined

26/06/27 13:15:56 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 388629 ms exceeds timeout 120000 ms
26/06/27 13:15:56 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/27 13:16:03 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at o

**Teoria:** `toPandas()` to akcja, która zbiera (collect) wszystkie dane na partycjach roboczych i przesyła je na węzeł główny (Driver), konwertując do struktury Pandas DataFrame. Uwaga: Można tego używać tylko na małych zbiorach (po limitowaniu np. top 10), w przeciwnym razie braknie pamięci RAM na Driverze!


---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StringType

# INICJALIZACJA SESJI SPARK:
spark = SparkSession.builder \
    .appName("Analiza_Przestepstw") \
    .master("local[*]") \
    .getOrCreate()

# 1. Wczytanie danych
df_crimes = spark.read.option("header", True).csv("chicago_crimes_sample.csv")
print("Surowe dane (5 wierszy):")
df_crimes.show(5, truncate=False)

# 2. Czyszczenie danych
df_cleaned = (
    df_crimes
    .dropDuplicates(["id"])
    .dropna(subset=["id", "date", "primary_type", "year", "location_description"])
    .withColumn("ParsedDate", F.to_timestamp(F.col("date"))) # <-- Tutaj wprowadzona zmiana, bez formatu
    .filter(F.col("ParsedDate").isNotNull())
    .withColumn("Hour", F.hour(F.col("ParsedDate")))
    .withColumn("year", F.col("year").cast("int"))
)

print(f"Liczba wierszy po czyszczeniu: {df_cleaned.count()}")

# 3. Klasyfikacja pory dnia
def classify_time_of_day(h):
    if h is None:
        return "nieznana"
    if 6 <= h < 12:
        return "rano"
    if 12 <= h < 18:
        return "dzień"
    if 18 <= h < 22:
        return "wieczór"
    return "noc"

time_of_day_udf = F.udf(classify_time_of_day, StringType())

df_cleaned = df_cleaned.withColumn("time_of_day", time_of_day_udf(F.col("Hour")))
df_cleaned = df_cleaned.cache()
df_cleaned.show(5)

# 4. Słownik kodów FBI
crime_type_dict = spark.createDataFrame([
    ("01A", "Homicide"),
    ("02", "Criminal Sexual Assault"),
    ("03", "Robbery"),
    ("04A", "Aggravated Assault"),
    ("04B", "Aggravated Battery"),
    ("05", "Burglary"),
    ("06", "Theft"),
    ("07", "Motor Vehicle Theft"),
    ("08A", "Arson"),
    ("08B", "Simple Assault"),
    ("09", "Other"),
], ["fbi_code", "fbi_category"])

df_enriched = (
    df_cleaned
    .join(broadcast(crime_type_dict), on="fbi_code", how="left")
)

# 5. Zapis wyniku jako Partycje Parquet
output_path = "chicago_crimes_parquet"
df_enriched.write.mode("overwrite").partitionBy("year").parquet(output_path)
print(f"Dane zapisane do: {output_path}/")

# 6. Agregacje i raporty
print("\n=== Top 15 typów przestępstw wg lokalizacji i pory dnia ===")
crime_by_location_time = (
    df_enriched
    .groupBy("location_description", "time_of_day", "primary_type")
    .agg(F.count("*").alias("crime_count"))
    .orderBy(F.desc("crime_count"))
)
crime_by_location_time.show(15, truncate=False)

print("\n=== Najpopularniejsze typy przestępstw w poszczególnych porach dnia ===")
(
    df_enriched
    .groupBy("time_of_day", "primary_type")
    .agg(F.count("*").alias("crime_count"))
    .orderBy("time_of_day", F.desc("crime_count"))
    .show(20, truncate=False)
)

print("\n=== Plan wykonania najcięższej agregacji (explain) ===")
crime_by_location_time.explain(mode="formatted")

# 7. Zwolnienie pamięci po zakończeniu analizy
df_cleaned.unpersist()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/27 13:51:33 WARN Utils: Your hostname, MacBook-Air-Remigiusz.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.6 instead (on interface en0)
26/06/27 13:51:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/27 13:51:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Surowe dane (5 wierszy):
+-------------+--------------+-----------------------+---------------------+----+------------+---------------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-----------------------+------------+-------------+--------+
|id           |case_number   |date                   |block                |iucr|primary_type|description                            |location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|updated_on             |latitude    |longitude    |location|
+-------------+--------------+-----------------------+---------------------+----+------------+---------------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-----------------------+------------+-------------+--------+
|14193193     |JK249780      |2026-05-07T00:00:00.000|057

26/06/27 13:51:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+-----------+--------------------+--------------------+----+-------------------+---------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------+-------------------+----+-----------+
|      id|case_number|                date|               block|iucr|       primary_type|    description|location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|          updated_on|    latitude|    longitude|location|         ParsedDate|Hour|time_of_day|
+--------+-----------+--------------------+--------------------+----+-------------------+---------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------+-------------------+----+-----------+
|14110975|   JK149364|2026-02-14T12:23:...|    0000X N STATE ST|086

26/06/27 13:51:41 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Dane zapisane do: chicago_crimes_parquet/

=== Top 15 typów przestępstw wg lokalizacji i pory dnia ===


+--------------------+-----------+-------------------+-----------+
|location_description|time_of_day|primary_type       |crime_count|
+--------------------+-----------+-------------------+-----------+
|STREET              |noc        |MOTOR VEHICLE THEFT|1068       |
|APARTMENT           |noc        |BATTERY            |1030       |
|STREET              |noc        |CRIMINAL DAMAGE    |873        |
|STREET              |wieczór    |MOTOR VEHICLE THEFT|771        |
|APARTMENT           |dzień      |BATTERY            |744        |
|STREET              |noc        |THEFT              |728        |
|STREET              |dzień      |MOTOR VEHICLE THEFT|627        |
|APARTMENT           |wieczór    |BATTERY            |625        |
|APARTMENT           |rano       |BATTERY            |620        |
|SMALL RETAIL STORE  |dzień      |THEFT              |591        |
|APARTMENT           |dzień      |THEFT              |589        |
|STREET              |dzień      |THEFT              |556     

+-----------+--------------------------------+-----------+
|time_of_day|primary_type                    |crime_count|
+-----------+--------------------------------+-----------+
|dzień      |THEFT                           |4307       |
|dzień      |BATTERY                         |2824       |
|dzień      |ASSAULT                         |1646       |
|dzień      |CRIMINAL DAMAGE                 |1305       |
|dzień      |OTHER OFFENSE                   |1154       |
|dzień      |DECEPTIVE PRACTICE              |1070       |
|dzień      |MOTOR VEHICLE THEFT             |881        |
|dzień      |BURGLARY                        |691        |
|dzień      |NARCOTICS                       |585        |
|dzień      |CRIMINAL TRESPASS               |426        |
|dzień      |WEAPONS VIOLATION               |276        |
|dzień      |ROBBERY                         |240        |
|dzień      |OFFENSE INVOLVING CHILDREN      |155        |
|dzień      |SEX OFFENSE                     |100       

DataFrame[id: string, case_number: string, date: string, block: string, iucr: string, primary_type: string, description: string, location_description: string, arrest: string, domestic: string, beat: string, district: string, ward: string, community_area: string, fbi_code: string, x_coordinate: string, y_coordinate: string, year: int, updated_on: string, latitude: string, longitude: string, location: string, ParsedDate: timestamp, Hour: int, time_of_day: string]

26/06/28 06:12:59 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1003718 ms exceeds timeout 120000 ms
26/06/28 06:12:59 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/28 06:30:11 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at 